# African Student Mobility to/from the USA — Exploratory Data Analysis

**Domain:** Global Immigration · **Sub-theme:** African student mobility · **Run date:** 2026-10-21

This notebook explores the SQLite warehouse built by the pipeline (`data/student_mobility.db`) for
15 African focus countries and the USA, using six World Bank WDI indicators:

| Key | WDI code | Meaning |
|---|---|---|
| `tertiary_enrollment` | SE.TER.ENRR | Gross tertiary enrollment ratio (%) |
| `education_expenditure` | SE.XPD.TOTL.GD.ZS | Government education spend (% GDP) |
| `gdp_per_capita` | NY.GDP.PCAP.CD | GDP per capita (current US$) |
| `remittances` | BX.TRF.PWKR.CD.DT | Personal remittances received (US$) |
| `net_migration` | SM.POP.NETM | Net migration (persons) |
| `population` | SP.POP.TOTL | Total population |

The **Student Mobility Pressure Index** (0–100) combines the tertiary-capacity gap, income gap vs. peers,
remittance intensity and net emigration rate — the push factors that drive outbound study, most
prominently toward the USA.

> Run the pipeline first: `python src/transformation/transform.py && python src/loading/load.py`

In [ ]:
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import seaborn as sns

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

# Resolve the project root whether the notebook runs from notebooks/ or the root.
ROOT = Path.cwd() if (Path.cwd() / "config").exists() else Path.cwd().parent
DB_PATH = ROOT / "data" / "student_mobility.db"
assert DB_PATH.exists(), f"Database not found at {DB_PATH} - run the pipeline first"
conn = sqlite3.connect(DB_PATH)

def q(sql: str) -> pd.DataFrame:
    return pd.read_sql_query(sql, conn)

q("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")

## 1. Data overview and quality

In [ ]:
countries = q("SELECT * FROM countries")
metrics = q("SELECT * FROM country_year_metrics")
obs = q("SELECT * FROM indicator_observations")
snapshot = q("SELECT * FROM mobility_snapshot")

print(f"Countries: {len(countries)} | country-years: {len(metrics)} | observations: {len(obs)}")
print(f"Year range: {metrics.year.min()}–{metrics.year.max()}")

coverage = (obs.groupby("indicator_key")
               .agg(observations=("value", "size"),
                    imputed=("is_imputed", "sum"),
                    first_year=("year", "min"),
                    last_year=("year", "max"),
                    countries=("iso3", "nunique")))
coverage["imputed_pct"] = (coverage.imputed / coverage.observations * 100).round(1)
coverage

In [ ]:
# Missingness of tertiary enrollment by country - a key caveat for education analysis.
pivot_missing = (metrics.pivot(index="iso3", columns="year", values="tertiary_enrollment")
                        .notna().astype(int))
fig, ax = plt.subplots(figsize=(13, 5))
sns.heatmap(pivot_missing, cmap=["#f2f2f2", "#2b8cbe"], cbar=False, linewidths=0.3, ax=ax)
ax.set_title("Tertiary enrollment data availability (blue = observed or interpolated)")
ax.set_xlabel("Year"); ax.set_ylabel("")
plt.tight_layout(); plt.show()

## 2. Visualization 1 — Student Mobility Pressure Index ranking

In [ ]:
rank_df = snapshot.dropna(subset=["mobility_pressure_index"]).sort_values("mobility_pressure_index")
palette = dict(zip(sorted(countries.sub_region.unique()), sns.color_palette("Set2", 6)))

fig, ax = plt.subplots(figsize=(10, 6.5))
bars = ax.barh(rank_df.country_name, rank_df.mobility_pressure_index,
               color=rank_df.sub_region.map(palette))
for bar, (_, row) in zip(bars, rank_df.iterrows()):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
            f"{row.mobility_pressure_index:.1f}  (enrol. data {int(row.tertiary_enrollment_year)})",
            va="center", fontsize=8)
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for r, c in palette.items() if r in set(rank_df.sub_region)]
labels = [r for r in palette if r in set(rank_df.sub_region)]
ax.legend(handles, labels, title="Sub-region", loc="lower right")
ax.set_xlim(0, 118)
ax.set_xlabel("Student Mobility Pressure Index (0–100, higher = stronger push to study abroad)")
ax.set_title(f"Student Mobility Pressure Index, {int(rank_df.mobility_pressure_index_year.max())}")
plt.tight_layout(); plt.show()
rank_df[["pressure_rank", "country_name", "sub_region", "mobility_pressure_index",
         "tertiary_enrollment", "gdp_pc_pct_of_usa"]].sort_values("pressure_rank")

## 3. Visualization 2 — Tertiary enrollment: Africa vs USA

In [ ]:
enrol = metrics.dropna(subset=["tertiary_enrollment"])
africa_median = (enrol[enrol.is_africa == 1].groupby("year").tertiary_enrollment.median())
usa_enrol = enrol[enrol.iso3 == "USA"].set_index("year").tertiary_enrollment

fig, ax = plt.subplots(figsize=(12, 6))
for iso3, grp in enrol[enrol.is_africa == 1].groupby("iso3"):
    ax.plot(grp.year, grp.tertiary_enrollment, color="grey", alpha=0.35, lw=1)
    ax.annotate(iso3, (grp.year.iloc[-1], grp.tertiary_enrollment.iloc[-1]), fontsize=7, color="grey")
ax.plot(africa_median.index, africa_median.values, color="#d95f02", lw=3, label="Africa-15 median")
ax.plot(usa_enrol.index, usa_enrol.values, color="#1b9e77", lw=3, label="USA")
ax.set_ylabel("Gross tertiary enrollment (%)"); ax.set_xlabel("Year")
ax.set_title("Tertiary enrollment: the capacity gap that drives outbound study")
ax.legend(); plt.tight_layout(); plt.show()

latest_common = enrol[enrol.iso3 == "USA"].year.max()
print(f"Latest USA enrollment year: {latest_common} -> {usa_enrol.loc[latest_common]:.1f}%")
print(f"Africa-15 median in {latest_common}: {africa_median.get(latest_common, np.nan):.1f}%")

## 4. Visualization 3 — Income gap vs tertiary capacity (interactive)

In [ ]:
# Latest year where each African country has both GDP pc and enrollment observed.
pairs = (metrics[(metrics.is_africa == 1)]
         .dropna(subset=["tertiary_enrollment", "gdp_per_capita", "population"])
         .sort_values("year").groupby("iso3").tail(1))
fig = px.scatter(
    pairs, x="gdp_per_capita", y="tertiary_enrollment", size="population",
    color="sub_region", text="iso3", log_x=True, size_max=55,
    hover_data={"country_name": True, "year": True, "gdp_pc_pct_of_usa": ":.2f"},
    labels={"gdp_per_capita": "GDP per capita, US$ (log)",
            "tertiary_enrollment": "Tertiary enrollment (%)"},
    title="Richer African countries enrol more students domestically (bubble = population)",
)
fig.update_traces(textposition="top center")
fig.update_layout(height=520)
fig.show()
print("Spearman rho (GDP pc vs enrollment):",
      round(pairs[["gdp_per_capita", "tertiary_enrollment"]].corr(method="spearman").iloc[0, 1], 3))

## 5. Visualization 4 — Remittances received (diaspora channel)

In [ ]:
remit = metrics[(metrics.is_africa == 1) & (metrics.year >= 2010)]
heat = remit.pivot(index="country_name", columns="year", values="remittances") / 1e9
heat = heat.loc[heat.mean(axis=1).sort_values(ascending=False).index]

fig, axes = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw={"width_ratios": [3, 2]})
sns.heatmap(heat, cmap="YlGnBu", annot=True, fmt=".1f", annot_kws={"size": 6},
            linewidths=0.3, cbar_kws={"label": "US$ bn"}, ax=axes[0])
axes[0].set_title("Personal remittances received (US$ bn)"); axes[0].set_xlabel("Year"); axes[0].set_ylabel("")

latest_r = (metrics[metrics.is_africa == 1].dropna(subset=["remittances_pct_gdp"])
            .sort_values("year").groupby("iso3").tail(1).sort_values("remittances_pct_gdp"))
axes[1].barh(latest_r.country_name, latest_r.remittances_pct_gdp, color="#41ab5d")
axes[1].set_xlabel("Remittances as % of GDP (latest)")
axes[1].set_title("Remittance dependence")
plt.tight_layout(); plt.show()

## 6. Visualization 5 — Net migration rates: Africa vs USA

In [ ]:
focus = ["NGA", "KEN", "GHA", "ETH", "EGY", "MAR", "ZAF", "USA"]
mig = metrics[metrics.iso3.isin(focus) & (metrics.year >= 2005)]
fig, ax = plt.subplots(figsize=(12, 6))
sns.lineplot(data=mig, x="year", y="net_migration_rate_per_1000", hue="country_name",
             style=mig.iso3.eq("USA").map({True: "USA", False: "Africa"}), lw=2, ax=ax)
ax.axhline(0, color="black", lw=0.8)
ax.axvspan(2019.5, 2021.5, color="orange", alpha=0.12, label="COVID-19 travel restrictions")
ax.set_ylabel("Net migration per 1,000 population"); ax.set_xlabel("Year")
ax.set_title("The USA is a persistent net receiver; most African focus countries are net senders")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
plt.tight_layout(); plt.show()

## 7. Visualization 6 — Education investment vs outcomes

In [ ]:
edu = (metrics.dropna(subset=["education_spend_per_capita_usd", "tertiary_enrollment"])
              .sort_values("year").groupby("iso3").tail(1))
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.scatterplot(data=edu, x="education_expenditure", y="tertiary_enrollment",
                hue="sub_region", s=120, ax=axes[0])
for _, r in edu.iterrows():
    axes[0].annotate(r.iso3, (r.education_expenditure, r.tertiary_enrollment),
                     xytext=(4, 4), textcoords="offset points", fontsize=8)
axes[0].set_title("Education effort (% GDP) vs tertiary enrollment")
axes[0].set_xlabel("Government education expenditure (% GDP)")
axes[0].set_ylabel("Tertiary enrollment (%)")

edu_sorted = edu.sort_values("education_spend_per_capita_usd")
axes[1].barh(edu_sorted.country_name, edu_sorted.education_spend_per_capita_usd,
             color=np.where(edu_sorted.iso3 == "USA", "#1b9e77", "#7570b3"))
axes[1].set_xscale("log")
axes[1].set_xlabel("Public education spend per capita, US$ (log scale)")
axes[1].set_title("Spend per person: USA vs Africa-15 (latest available)")
plt.tight_layout(); plt.show()

## 8. Visualization 7 — Correlation of mobility drivers (African countries)

In [ ]:
cols = ["tertiary_enrollment", "education_expenditure", "gdp_per_capita",
        "remittances_pct_gdp", "net_migration_rate_per_1000", "mobility_pressure_index"]
corr = metrics[metrics.is_africa == 1][cols].corr(method="spearman")
fig, ax = plt.subplots(figsize=(8, 6.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            square=True, ax=ax)
ax.set_title("Spearman correlation of mobility drivers (Africa-15, all years)")
plt.tight_layout(); plt.show()

## 9. Pressure index over time (interactive)

In [ ]:
trend = metrics[(metrics.is_africa == 1) & metrics.mobility_pressure_index.notna() & (metrics.year >= 2010)]
fig = px.line(trend, x="year", y="mobility_pressure_index", color="country_name",
              line_group="iso3", markers=True,
              labels={"mobility_pressure_index": "Pressure index", "year": "Year"},
              title="Student Mobility Pressure Index trajectories, 2010+")
fig.update_layout(height=500)
fig.show()

## 10. Running the SQL layer from Python

In [ ]:
# Query Q8 from sql/queries.sql: income convergence vs the USA since 2015.
convergence = q('''
WITH traj AS (
    SELECT m.iso3, m.country_name, m.year, m.gdp_pc_pct_of_usa,
           FIRST_VALUE(m.gdp_pc_pct_of_usa) OVER w AS pct_usa_2015,
           LAST_VALUE(m.gdp_pc_pct_of_usa)  OVER w AS pct_usa_latest
    FROM country_year_metrics m
    WHERE m.is_africa = 1 AND m.year >= 2015 AND m.gdp_pc_pct_of_usa IS NOT NULL
    WINDOW w AS (PARTITION BY m.iso3 ORDER BY m.year
                 ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING)
)
SELECT DISTINCT iso3, country_name,
       ROUND(pct_usa_2015, 2) AS pct_usa_2015,
       ROUND(pct_usa_latest, 2) AS pct_usa_latest,
       ROUND(pct_usa_latest - pct_usa_2015, 2) AS change_pp
FROM traj ORDER BY change_pp DESC
''')
convergence

## 11. Key computed insights

In [ ]:
latest_year = int(snapshot.mobility_pressure_index_year.max())
top3 = snapshot.nsmallest(3, "pressure_rank")
bottom = snapshot.nlargest(1, "pressure_rank").iloc[0]
usa_last = metrics[(metrics.iso3 == "USA")].dropna(subset=["tertiary_enrollment"]).iloc[-1]
remit_by_country = snapshot.set_index("country_name").remittances.sort_values(ascending=False)
share_top2 = remit_by_country.iloc[:2].sum() / remit_by_country.sum() * 100
diverging = (convergence.change_pp < 0).sum()

print(f"1. Highest pressure ({latest_year}): " + ", ".join(
    f"{r.country_name} ({r.mobility_pressure_index:.1f})" for _, r in top3.iterrows()))
print(f"2. Lowest pressure: {bottom.country_name} ({bottom.mobility_pressure_index:.1f})")
print(f"3. USA tertiary enrollment {int(usa_last.year)}: {usa_last.tertiary_enrollment:.1f}% vs "
      f"Africa-15 median {africa_median.get(int(usa_last.year), np.nan):.1f}%")
print(f"4. {remit_by_country.index[0]} and {remit_by_country.index[1]} receive "
      f"{share_top2:.0f}% of the Africa-15 remittance total (latest values)")
print(f"5. {diverging} of {len(convergence)} countries lost ground vs USA GDP per capita since 2015")

## 12. Conclusions

* **Capacity gap is the main push factor.** Across the 15 focus countries, gross tertiary enrollment
  sits far below the USA benchmark; East African countries (Ethiopia, Uganda, Tanzania, Rwanda) show the
  highest Student Mobility Pressure, while South Africa, Morocco and Tunisia show the lowest.
* **Income and enrollment move together** (strong positive rank correlation), so slow income
  convergence with the USA keeps pressure for overseas study high.
* **Diaspora finance is concentrated** — Egypt and Nigeria dominate remittance inflows, a channel that
  partly funds study abroad and reflects established migration corridors to the USA.
* **Migration asymmetry** — the USA remains a structural net receiver (with a COVID-era dip in
  2020–21) while most African focus countries are net senders.
* **Data caveat** — tertiary enrollment is sparse for Nigeria, Ethiopia, Uganda and Zambia; the
  pressure index carries the last observed value forward and records the reference year explicitly.

In [ ]:
conn.close()